# IM-Benchmark-Lab — Google Colab Setup

Notebook này cài đặt và chạy dự án **IM-Benchmark-Lab** trên Google Colab.

## Nội dung

1. Kiểm tra môi trường
2. Cài system dependencies
3. Lấy source code
4. Cài thư viện Python
5. Build C++ core (pybind11)
6. Smoke test Python ↔ C++
7. Chạy unit tests
8. Chạy thí nghiệm mẫu
9. Tùy chọn: tải trước dữ liệu
10. Tùy chọn: GUI smoke test
11. Tùy chọn: lưu kết quả vào Google Drive
12. Chạy lại thí nghiệm với tham số tùy chỉnh

## Cách dùng

- Chọn **Runtime → Change runtime type → CPU** (dự án không cần GPU).
- Chạy lần lượt các cell từ trên xuống.
- Lần chạy đầu có thể mất 3–10 phút để cài thư viện và build C++.

> **Lưu ý:** Colab không có desktop thật, nên GUI CustomTkinter không mở được trực tiếp. Notebook chỉ chạy GUI smoke test bằng Xvfb. Muốn dùng GUI, chạy trên WSL2/Windows.


## 0. Kiểm tra môi trường


In [1]:
!python3 --version
!cmake --version | head -n 1
!g++ --version | head -n 1
!nproc
!df -h /content | tail -n 1


Python 3.13.16
cmake version 3.31.10
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
2
overlay         113G   83M  113G   1% /


## 1. Cài system dependencies

Cài `git`, `cmake`, `g++`, Python headers, Tk và OpenMP runtime.


In [4]:
%%bash
set -Eeuo pipefail

export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq \
    git \
    cmake \
    build-essential \
    g++ \
    python3-dev \
    python3-venv \
    python3-tk \
    libgomp1

echo "System dependencies OK"
cmake --version | head -n 1
g++ --version | head -n 1
python3 --version


Selecting previously unselected package tk8.6-blt2.5.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../00-tk8.6-blt2.5_2.5.3+dfsg-7build1_amd64.deb ...
Unpacking tk8.6-blt2.5 (2.5.3+dfsg-7build1) ...
Selecting previously unselected package blt.
Preparing to unpack .../01-blt_2.5.3+dfsg-7build1_amd64.deb ...
Unpacking blt (2.5.3+dfsg-7build1) ...
Selecting previously unselected package javascript-common.
Preparing to unpack .../02-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libjs-underscore.
Preparing to unpack .../03-libjs-underscore_1.13.4~dfsg+~1.11.4-3_all.deb ...
Unpacking libjs-underscore (1.13.4~dfsg+~1.11.4-3) ...
Selecting previously unselected package libjs-sphinxdoc.
Preparing to unpack .../04-libjs-sphinxdoc_7.2.6-6_all.deb ...
Unpacking libjs-sphinxdoc (7.2.6-6) ...
Selecting previously unselected package python3.12-dev.
Preparing to unpack .../05-pyth

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Lấy source code

Clone repo (hoặc dùng lại nếu đã có).


In [5]:
%%bash
set -Eeuo pipefail

REPO_URL="https://github.com/SLMR05-05/influence-maximization-benchmark.git"
BRANCH="main"
PROJECT_DIR="/content/influence-maximization-benchmark"

if [ -f "$PROJECT_DIR/pyproject.toml" ] && [ -d "$PROJECT_DIR/src/im_research" ]; then
    echo "Đã có source: $PROJECT_DIR"
else
    git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
fi

git config --global --add safe.directory "$PROJECT_DIR" || true
mkdir -p "$PROJECT_DIR/data/raw" "$PROJECT_DIR/data/processed" "$PROJECT_DIR/runs"
cd "$PROJECT_DIR"
echo "Commit: $(git rev-parse --short HEAD 2>/dev/null || echo 'n/a')"


Đã có source: /content/influence-maximization-benchmark
Commit: e4b3aa0


## 3. Cài thư viện Python

Colab đã có sẵn Python runtime; cài thư viện trực tiếp vào runtime hiện tại.


In [7]:
%%bash
set -Eeuo pipefail

PROJECT_DIR="/content/influence-maximization-benchmark"

python -m pip install --upgrade pip setuptools wheel
python -m pip install --no-cache-dir -r "$PROJECT_DIR/requirements.txt"
python -m pip install --no-cache-dir --no-build-isolation -e "$PROJECT_DIR"

python --version
echo "Python dependencies OK"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 21.2 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.4/554.4 kB 88.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 176.8 MB/s  0:00:00

Obtaining file:///content/influence-maximization-benchmark
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for im_research (pyproj

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  /usr/local/lib/python3.13/dist-packages/setuptools/config/_apply_pyprojecttoml.py:82: BuildBackendWarning: `project.license` as a TOML table is deprecated
  !!
  
          ********************************************************************************
          Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
  
          By 2027-Feb-18, you need to update your project and remove deprecated calls
          or your builds will no longer be supported.
  
          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
          *********************************************************************

## 4. Build C++ core

Build `_im_core*.so` và đặt trực tiếp vào `src/im_research/`.


In [8]:
%%bash
set -Eeuo pipefail

PROJECT_DIR="/content/influence-maximization-benchmark"
cd "$PROJECT_DIR"

rm -rf build
mkdir -p build
cd build

cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CXX_FLAGS="-O3 -march=native -fopenmp"

make -j"$(nproc)"

cd "$PROJECT_DIR"
find src/im_research -maxdepth 1 -name '_im_core*.so' -ls


-- The CXX compiler identification is GNU 13.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- OpenMP found! Enabling multi-threaded simulations.
-- Found PythonInterp: /usr/bin/python3 (found suitable version "3.13.16", minimum required is "3.9")
-- Found PythonLibs: /usr/lib/x86_64-linux-gnu/libpython3.13.so
-- Performing Test HAS_FLTO_AUTO
-- Performing Test HAS_FLTO_AUTO - Success
-- Found pybind11: /usr/local/lib/python3.13/dist-packages/pybind11/include (found version "3.1.0")
-- pybind11 found! Building Python extension module '_im_core'.
-- Configuring done (1.1s)
-- Generating done (0.0s)
-- Build files have been written to: /content/influence-maximization-benchmark/build
[ 12%] Building CXX object CMakeFiles/im_core_l

CMake Warning (dev) at /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/FindPythonLibsNew.cmake:101 (message):
  Policy CMP0148 is not set: The FindPythonInterp and FindPythonLibs modules
  are removed.  Run "cmake --help-policy CMP0148" for policy details.  Use
  the cmake_policy command to set the policy and suppress this warning, or
  preferably upgrade to using FindPython, either by calling it explicitly
  before pybind11, or by setting PYBIND11_FINDPYTHON ON before pybind11.
Call Stack (most recent call first):
  /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/pybind11Tools.cmake:44 (find_package)
  /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/pybind11Common.cmake:243 (include)
  /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/pybind11Config.cmake:257 (include)
  CMakeLists.txt:51 (find_package)
This warning is for project developers.  Use -Wno-dev to suppress it.



## 5. Smoke test Python ↔ C++


In [9]:
%%bash
set -Eeuo pipefail
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python - <<'PY'
import im_research
from im_research import _im_core

print(f"im_research : {im_research.__file__}")
print(f"C++ core    : {_im_core.__file__}")

g = _im_core.Graph.from_edge_list(
    3, [(0, 1), (1, 2)], [0.5, 0.5], False
)
print(f"Graph test  : {g.num_nodes()} đỉnh, {g.num_edges()} cạnh")

ev = _im_core.Evaluator(
    g, _im_core.DiffusionModel.IndependentCascade,
    num_mc_simulations=50, seed=42,
)
print(f"IC spread   : {ev.evaluate([0])}")
PY


im_research : /content/influence-maximization-benchmark/src/im_research/__init__.py
C++ core    : /content/influence-maximization-benchmark/src/im_research/_im_core.cpython-313-x86_64-linux-gnu.so
Graph test  : 3 đỉnh, 4 cạnh
IC spread   : 1.8


## 6. Chạy unit tests


In [10]:
%%bash
set -Eeuo pipefail
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python -m pytest tests/unit -v


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/influence-maximization-benchmark
configfile: pyproject.toml
plugins: cov-7.1.0, anyio-4.16.0, platformdirs-4.14.3, langsmith-0.14.17, typeguard-4.6.0
collecting ... collected 27 items

tests/unit/test_algorithms.py::test_degree_centrality PASSED             [  3%]
tests/unit/test_algorithms.py::test_degree_discount PASSED               [  7%]
tests/unit/test_algorithms.py::test_pagerank PASSED                      [ 11%]
tests/unit/test_algorithms.py::test_celf PASSED                          [ 14%]
tests/unit/test_algorithms.py::test_baseline_ga PASSED                   [ 18%]
tests/unit/test_algorithms.py::test_iga PASSED                           [ 22%]
tests/unit/test_celf.py::test_celf_basic PASSED                          [ 25%]
tests/unit/test_celf.py::test_celf_unit_costs_equal

## 7. Chạy thí nghiệm mẫu

Chạy 6 thuật toán trên Karate Club với `k=10`, `R_search=200`, `R_val=1000`.


In [11]:
%%bash
set -Eeuo pipefail
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python -m im_research.experiments.runner \
    --dataset "Karate Club" \
    --model IC \
    --p 0.1 \
    --k 10 \
    --mc 200 \
    --mc_val 1000 \
    --runs 1 \
    --algo iga \
    --algo ga \
    --algo celf \
    --algo degree_discount \
    --algo pagerank \
    --algo degree


      IM-Benchmark-Lab — INFLUENCE MAXIMIZATION EXPERIMENT RUNNER
Dataset      : Karate Club
Model        : IC (uniform p=0.1)
Budget k     : 10
MC Search (R): 200
MC Val (R)   : 1000
Seeds        : search=42, validation=12345
Repetitions  : 1
Algorithms   : iga, ga, celf, degree_discount, pagerank, degree
----------------------------------------------------------------------
[1/4] Loading graph dataset 'Karate Club' (Model: IC)...
      Graph loaded: 34 nodes, 156 edges

[2/4] Running 6 algorithms across 1 repetition(s)...

>> Executing: IGA
   ── [Run 1/1] Khởi chạy IGA (seed=42)...
      [IGA] Pha 1: Đang trích xuất tri thức topo (Louvain & PageRank)...
      [IGA] Pha 1: Phân hoạch 3 cộng đồng; Hạn ngạch phần dư lớn nhất: [5, 4, 1]... (Tổng k = 10)
      [IGA] Pha 2: Khởi tạo quần thể N=50 (80% Topo-guided, 20% Random) & đánh giá hàm LIE-2 ban đầu...
      [IGA] Pha 2: Hoàn tất t=0 | Nghiệm tốt nhất f_LIE = 15.06 | Đa dạng Jaccard = 0.781
      [IGA] Pha 3: Bắt đầu chu trình tiến h

<frozen runpy>:130: RuntimeWarning: 'im_research.experiments.runner' found in sys.modules after import of package 'im_research.experiments', but prior to execution of 'im_research.experiments.runner'; this may result in unpredictable behaviour


## 8. Tùy chọn — tải trước 5 bộ dữ liệu

`data/raw/` không được commit lên Git. Cell này tải trước để lần chạy sau nhanh hơn.


In [12]:
%%bash
set -Eeuo pipefail
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

python - <<'PY'
from im_research.data.loader import load_networkx_graph

datasets = [
    "Karate Club",
    "Dolphins",
    "ego-Facebook",
    "NetHEPT",
    "Email-Enron",
]
for name in datasets:
    try:
        g, meta = load_networkx_graph(name)
        print(f"{name:15s} {g.number_of_nodes():>6} đỉnh / {g.number_of_edges():>7} cạnh")
    except Exception as exc:
        print(f"{name:15s} LỖI: {exc}")
PY


Karate Club         34 đỉnh /      78 cạnh
Dolphins            62 đỉnh /     159 cạnh
[DataLoader] Downloading facebook dataset from SNAP...
[DataLoader] Extracting facebook.txt.gz...
ego-Facebook      4039 đỉnh /   88234 cạnh
[DataLoader] Downloading NetHEPT dataset from https://raw.githubusercontent.com/Wang-GY/IMP/master/NetHEPT.txt...
NetHEPT          15233 đỉnh /   31376 cạnh
[DataLoader] Downloading enron dataset from SNAP...
[DataLoader] Extracting enron.txt.gz...
Email-Enron      36692 đỉnh /  183831 cạnh


## 9. Tùy chọn — GUI smoke test (Xvfb)

Colab không có desktop; cell này chỉ kiểm tra GUI khởi động được, không hiển thị cửa sổ.


In [13]:
%%bash
set -Eeuo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get install -y -qq xvfb xauth

cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

xvfb-run -a -s "-screen 0 1400x900x24" python - <<'PY'
from im_research.gui.app import IMApp

app = IMApp()
app.after(3000, app.destroy)
app.mainloop()
print("GUI smoke test OK")
PY


GUI smoke test OK


## 10. Tùy chọn — lưu kết quả vào Google Drive

Cell này dùng `google.colab`, chỉ chạy được trên Colab.


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

src = Path('/content/influence-maximization-benchmark/runs')
dst = Path('/content/drive/MyDrive/IM-Benchmark-Lab/runs')
dst.parent.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print('Đã lưu kết quả vào', dst)
else:
    print('Chưa có thư mục runs/. Hãy chạy thí nghiệm trước.')


Mounted at /content/drive
Đã lưu kết quả vào /content/drive/MyDrive/IM-Benchmark-Lab/runs


## 11. Chạy lại thí nghiệm với tham số tùy chỉnh

Sửa các biến ở đầu cell rồi chạy.


In [15]:
%%bash
set -Eeuo pipefail
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"

# ── Tham số có thể sửa ──────────────────────────────────────
DATASET="Karate Club"
MODEL="IC"
P="0.1"
K="10"
MC="200"
MC_VAL="1000"
RUNS="3"
ALGOS="iga ga celf degree_discount pagerank degree"
# ────────────────────────────────────────────────────────────

ALGO_ARGS=""
for a in $ALGOS; do
    ALGO_ARGS="$ALGO_ARGS --algo $a"
done

python -m im_research.experiments.runner \
    --dataset "$DATASET" \
    --model "$MODEL" \
    --p "$P" \
    --k "$K" \
    --mc "$MC" \
    --mc_val "$MC_VAL" \
    --runs "$RUNS" \
    $ALGO_ARGS


      IM-Benchmark-Lab — INFLUENCE MAXIMIZATION EXPERIMENT RUNNER
Dataset      : Karate Club
Model        : IC (uniform p=0.1)
Budget k     : 10
MC Search (R): 200
MC Val (R)   : 1000
Seeds        : search=42, validation=12345
Repetitions  : 3
Algorithms   : iga, ga, celf, degree_discount, pagerank, degree
----------------------------------------------------------------------
[1/4] Loading graph dataset 'Karate Club' (Model: IC)...
      Graph loaded: 34 nodes, 156 edges

[2/4] Running 6 algorithms across 3 repetition(s)...

>> Executing: IGA
   ── [Run 1/3] Khởi chạy IGA (seed=42)...
      [IGA] Pha 1: Đang trích xuất tri thức topo (Louvain & PageRank)...
      [IGA] Pha 1: Phân hoạch 3 cộng đồng; Hạn ngạch phần dư lớn nhất: [5, 4, 1]... (Tổng k = 10)
      [IGA] Pha 2: Khởi tạo quần thể N=50 (80% Topo-guided, 20% Random) & đánh giá hàm LIE-2 ban đầu...
      [IGA] Pha 2: Hoàn tất t=0 | Nghiệm tốt nhất f_LIE = 15.06 | Đa dạng Jaccard = 0.781
      [IGA] Pha 3: Bắt đầu chu trình tiến h

<frozen runpy>:130: RuntimeWarning: 'im_research.experiments.runner' found in sys.modules after import of package 'im_research.experiments', but prior to execution of 'im_research.experiments.runner'; this may result in unpredictable behaviour


In [19]:
Python main_app.py

SyntaxError: invalid syntax (2728242692.py, line 1)

## Ghi chú

- **GUI:** Colab không có desktop. Muốn dùng GUI thật, chạy `python src/im_research/gui/main_app.py` trên WSL2/Windows.
- **CLI:** dùng `python -m im_research.experiments.runner`; entry point `im-bench` trong `pyproject.toml` hiện trỏ tới module chưa tồn tại.
- **Dữ liệu:** `data/raw/` không nằm trong Git; loader tự tải Facebook, Enron, NetHEPT từ Internet. Karate lấy từ NetworkX; Dolphins tải từ UCI.
- **Kết quả:** `runs/EXP_<timestamp>/summary.json` và `runs/latest_results.json`.
- **Cập nhật code:** `!git -C /content/influence-maximization-benchmark pull --ff-only origin main`
- **Cấu hình IC:** `p=0.1`, `R_search=200`, `R_val=1000`; đổi bằng `--p`, `--mc`, `--mc_val`.
